# ☁️ Manhwa Story Studio Pro — Cloud T4 GPU Worker (GitHub Sync)

**Before running:** Go to **Runtime → Change runtime type → `T4 GPU` → Save**.

- **Scriptwriting:** Runs on your local PC (Gemini / OpenRouter / Groq)
- **Voices:** Local `SPRINGLab/F5-Hindi-24KHz` on Colab T4 (Zero Gemini API calls)
- **Video Rendering:** Hybrid Auto-Director + `h264_nvenc` on Colab T4
- **Auto-Sync:** Finished `.mp4` + locked voices automatically download to your local PC AND back up to Google Drive.

In [ ]:
# =====================================================================
# CELL 1: Mount Google Drive (auto-backup) & Pull Latest Code from GitHub
# =====================================================================
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO_OWNER = "sahilpatil202001-bit"
REPO_NAME = "Manhwa-Story-Studio"
BRANCH = "main"
STUDIO_DIR = Path("/content/studio")

# If your repo is PRIVATE, add a secret named GITHUB_TOKEN in Colab's left-sidebar Key icon (🔑).
# If your repo is PUBLIC, leave it empty — no token needed!
gh_token = ""
try:
    from google.colab import userdata
    gh_token = (userdata.get("GITHUB_TOKEN") or "").strip()
except Exception:
    pass

if gh_token:
    repo_url = f"https://{gh_token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
else:
    repo_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

if (STUDIO_DIR / ".git").exists():
    print("Pulling latest updates from GitHub...")
    !git -C /content/studio fetch origin
    !git -C /content/studio reset --hard origin/{BRANCH}
else:
    print("Cloning repository from GitHub...")
    !rm -rf /content/studio
    !git clone --depth 1 -b {BRANCH} {repo_url} /content/studio

%cd /content/studio
!ls -lh exports/*.wav exports/voice_refs/*.wav

In [ ]:
# =====================================================================
# CELL 2: Install Dependencies, Verify T4 GPU & Start Cloud Render Worker
# =====================================================================
%cd /content/studio

!apt-get -qq update && apt-get -qq install -y ffmpeg fonts-dejavu-core fonts-noto-core
!pip install -q f5-tts
!pip install -q -e .
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

print("\n--- GPU CHECK (Must show Tesla T4 / L4) ---")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import os
os.environ.pop("GEMINI_API_KEY", None)
os.environ.pop("GEMINI_API_KEYS", None)
os.environ["TTS_PROVIDER"] = "f5_hindi"
os.environ["HF_HOME"] = "/content/.cache/huggingface"
os.environ["TORCH_HOME"] = "/content/.cache/torch"
os.environ["STUDIO_FONT_LATIN"] = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
os.environ["STUDIO_FONT_HINDI"] = "/usr/share/fonts/truetype/noto/NotoSansDevanagari-Regular.ttf"
os.environ["STUDIO_DISABLE_GPU"] = "0"
os.environ["STUDIO_PANEL_LAYOUT"] = "hybrid"

# Restart worker cleanly if already running
!pkill -f "manhwa_studio.dashboard.cloud_worker" 2>/dev/null || true
!PYTHONPATH=/content/studio/src STUDIO_CLOUD_TOKEN="manhwa_t4_secret_2026" nohup python -m manhwa_studio.dashboard.cloud_worker --port 8766 > /tmp/cloud_worker.log 2>&1 &
!sleep 2
!cat /tmp/cloud_worker.log

In [ ]:
# =====================================================================
# CELL 3: Open Stable HTTP/2 Tunnel (Leave This Cell Running!)
# Copy the https://xxxx.trycloudflare.com URL into your Local Dashboard
# Token: manhwa_t4_secret_2026
# =====================================================================
!cloudflared tunnel --protocol http2 --url http://127.0.0.1:8766